# Gradients
Since QAOA is a variational quantum algorithm, the trainability of the circuit is related to the behavior of the gradients of the cost function with respect to the variational parameters $\vec{\gamma}$ and $\vec{\beta}$.
Large gradient magnitudes correspond to a rapidly varying cost-function landscape, which facilitates optimization.
In contrast, small gradients are associated with barren plateaus that is regions where the cost function is flat and optimization more difficult.
For a given formula, the cost function is implemented in python as follows:

In [ ]:
sparse_pauli_list = sparse_pauli_list_ksat(formula)
cost_hamiltonian = SparsePauliOp.from_list(sparse_pauli_list)
        
raw_ansatz = QAOAAnsatz(cost_hamiltonian, reps=p)
ansatz = transpile(raw_ansatz, backend=backend, optimization_level=1)

def cost_function(params):
    pub = (ansatz, cost_hamiltonian, params)
    job = estimator.run([pub])
    result = job.result()[0]
    return result.data.evs

Because gradients average to zero over random parameter configurations, we characterize their typical magnitude using the standard deviation (SD) of the gradients magnitude rather than their mean.
Gradients are evaluated at 5000 randomly sampled circuit parameters in $[0,2\pi]^{2p}$ using a numerical finite-difference method implemented in scipy.
For describing the cost-function landscape as a function of m/n, it is reasonable to expect that the partial derivative of the gradient w.r.t. one coordinate can as well be used instead of the gradient magnitude.
To compute the gradient magnitude at a random point:

In [ ]:
params = np.random.uniform(low=0, high=2*np.pi, size=ansatz.num_parameters)
grad = approx_fprime(params, cost_function)

Instead, to compute the partial derivative w.r.t., wlog, $\gamma_1$:

In [ ]:
params = np.random.uniform(low=0, high=2*np.pi, size=ansatz.num_parameters)

def f1(x):
    p_copy = params.copy()
    p_copy[0] = x[0]
    return cost_function(p_copy)

partial = approx_fprime(np.array([params[0]]), f1)

We compute the gradient magnitude/partial derivative SD on 5000 points, and average it over some iterations (50 or 100, see later).
For visualization purposes, we plot the average of the inverse SD.

plots...

Larger values indicate smaller typical gradients and therefore more difficult optimization.
We see that, when the QAOA depth $p$ is small, this peak is no visible; however, in this regime QAOA also fails to produce accurate solutions, making trainability irrelevant.
For large $p$, peaks are clearly visible.
For 2-sat, the peak is at a clause-to-variable ratio larger than the SAT-UNSAT phase transition ratio.
For 3-sat, the peak is at a smaller clause-to-variable ratio than the SAT-UNSAT transition ratio.
Recall that the SAT-UNSAT phase transition ratio corresponds to the region of maximum empirical hardness for classical algorithms.
Thus we see that the empirical hardness for QAOA is different from classical algorithms.
Although this result holds for SAT problem, we expect the __computational__ phase transition in QAOA to apply to all
combinatorial optimization problems. In particular, as 3-SAT is NP-complete, the clause-to-variable ratio represents a universal characterization of a ‘problem density’, and the computational phase transition applies to all NP-complete problems in this regard. 